# SimBench grids in eDisGo

This notebook shows how to analyse the [SimBench](https://simbench.de) benchmark
grids with eDisGo. It covers

1. what SimBench is and what it is used for,
2. how to get a SimBench grid,
3. how eDisGo imports it and what the import maps to what,
4. what is in a SimBench grid: topology, heat pumps, electric vehicles, batteries
   and their time series,
5. power flow analysis and grid issues,
6. a comparison of the 2016 grid (scenario 0) with the 2034 scenario (scenario 2),
7. grid reinforcement and its costs,
8. what else you can do with SimBench grids in eDisGo, and where the limits are.

## 1 What is SimBench?

Research on distribution grids needs grid data, but real grid data of
distribution system operators is confidential. Every study therefore used its own
grids, and results of different methods could hardly be compared or reproduced.

**SimBench** ("Simulationsdatensatz für Benchmark-Netze", 2015–2019, Universität
Kassel, Fraunhofer IEE, RWTH Aachen and TU Dortmund) closes this gap. It is an
open benchmark data set of *synthetic but realistic* German power grids, derived
from real grid data:

* **All voltage levels**: extra high (EHV), high (HV), medium (MV) and low
  voltage (LV). Grids of neighbouring levels can be combined, e.g. one MV grid
  with all LV grids connected to it.
* **Grid types**: MV grids `rural`, `semiurb`, `urban` and `comm` (commercial),
  LV grids `rural1`–`rural3`, `semiurb4`, `semiurb5` and `urban6`.
* **Time series**: one year of 15-minute profiles for every load, generator and
  storage unit.
* **Three scenarios** of the energy transition:

  | scenario | year  | heat pumps | home charging points | PV with home battery | PV       |
  |----------|-------|------------|----------------------|-----------|----------|
  | 0        | 2016  | –          | –                    | –         | 349 MW   |
  | 1        | 2024  | 1 778      | 1 080                | 3 746     | 689 MW   |
  | 2        | 2034  | 4 836      | 3 001                | 6 533     | 971 MW   |

  (numbers for the complete SimBench data set)

Scenario 0 is the grid as it was when SimBench was built, with profiles for 2016.
SimBench designed these grids to keep all voltage and loading limits, also in the
(n-1) case; the scenarios 1 and 2 are meant to violate them.

The grids are meant as common ground to compare methods for grid planning,
operation, state estimation, flexibility use etc. and to make results
reproducible.

### SimBench codes

Every grid is identified by a code, e.g. `1-MVLV-rural-all-2-sw`:

| part     | meaning                                                        |
|----------|----------------------------------------------------------------|
| `1`      | SimBench version                                               |
| `MVLV`   | voltage levels: MV grid with underlying LV grids               |
| `rural`  | type of the upper grid                                         |
| `all`    | which LV grids are modelled in detail (`all`, or one, e.g. `1.108`) |
| `2`      | scenario                                                       |
| `sw`     | switches are modelled                                          |

LV grids that are not modelled in detail are represented at their MV bus by one
load, one generator and, in scenarios 1 and 2, one PV system with home battery.

### And SimBench *Sektor*?

The follow-up project *SimBench Sektor* (2023–2026, Hochschule Bonn-Rhein-Sieg,
Fraunhofer IEE, Gas- und Wärme-Institut Essen) extends SimBench to gas and heating
grids. Its data had not been published by November 2026. The sector coupling on
the electricity side, i.e. heat pumps and electric vehicles, is already part of
the SimBench scenarios 1 and 2, and this is what eDisGo imports.

## 2 Why SimBench in eDisGo?

eDisGo usually works on [ding0](https://github.com/openego/ding0) grids with
scenario data from the OpenEnergy Platform. SimBench grids add

* **well-known benchmark grids**, so eDisGo results can be compared with other
  tools and published studies,
* **complete time series without any database access**,
* **future scenarios** with heat pumps, home charging points and home batteries,
* **different grid types** (rural to urban) with the same data structure.

## 3 Getting a SimBench grid

eDisGo reads the csv folder of one SimBench grid. There are two ways to get it:

**a) Download** the grid as `csv` from https://simbench.de/de/datensaetze/ and
unzip it.

**b) Extract it with the `simbench` python package.** `simbench` needs newer
pandas versions than eDisGo, so install it in a separate environment
(`pip install simbench`) and run:

```python
import os
from simbench.converter.read_and_write import write2csv
from simbench.networks.extract_simbench_grids_from_csv import (
    complete_data_path, get_extracted_csv_data, get_relevant_subnets)
from simbench.networks.profiles import filter_unapplied_profiles
from simbench.networks.simbench_code import get_simbench_code_and_parameters

code = "1-MVLV-rural-all-2-sw"
_, parameters = get_simbench_code_and_parameters(code)
input_path = complete_data_path(parameters[5])
data = get_extracted_csv_data(get_relevant_subnets(parameters, input_path), input_path)
filter_unapplied_profiles(data)
write2csv(code, {name: table for name, table in data.items() if len(table)})
```

This notebook uses the rural MV grid with all its LV grids in scenario 0 and 2.
Set `SIMBENCH_DIR` to the folder that holds the grid folders.

In [ ]:
import logging
import os

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.collections import LineCollection

from edisgo import EDisGo
from edisgo.flex_opt import check_tech_constraints

for logger in ["edisgo", "pypsa"]:
    logging.getLogger(logger).setLevel(logging.ERROR)
plt.rcParams["figure.dpi"] = 110

SIMBENCH_DIR = os.environ.get("SIMBENCH_DIR", "simbench_grids")
GRID = "1-MVLV-rural-all-{scenario}-sw"

## 4 Importing a SimBench grid

`EDisGo.import_simbench_grid` reads the grid and, by default, sets the time series
of all loads, generators and storage units from the SimBench profiles. A full year
of a large grid needs a lot of memory, so we select two weeks: one in winter, when
heat pumps run most, and one in summer with high PV feed-in.

SimBench profiles are given for 2016. Like all time series in eDisGo, they are
relabelled to the reference year (2011 by default), keeping each date; 29 February
is left out.

In [ ]:
winter = pd.date_range("2011-01-10", periods=7 * 96, freq="15min")
summer = pd.date_range("2011-07-04", periods=7 * 96, freq="15min")
timeindex = winter.append(summer)

edisgo = EDisGo()
edisgo.import_simbench_grid(
    os.path.join(SIMBENCH_DIR, GRID.format(scenario=2)), timeindex=timeindex
)
edisgo.topology

### What the import maps to what

| SimBench                                   | eDisGo                                                       |
|--------------------------------------------|--------------------------------------------------------------|
| nodes connected by closed switches         | one bus                                                      |
| open switches at line ends (ring splitting points) | switch disconnectors (`topology.switches_df`)        |
| open switches that are the only connection of a grid part to the HV/MV substation (separately operated busbars) | closed, as eDisGo models one MV station |
| HV/MV transformers                         | `topology.transformers_hvmv_df`, the HV side is left out     |
| MV/LV transformers                         | `topology.transformers_df`                                   |
| LV grid `LV1.108`                          | LV grid with ID `1108` (the MV grid `MV1.101` gets ID `1101`)|
| loads with profiles `Air_*`, `Soil_*`      | loads of type `heat_pump`                                    |
| loads with profiles `HLS_*`                | loads of type `charging_point`, sector `home`                |
| all other loads                            | `conventional_load` (sector from the profile: `H0` residential, `G*` cts, `L*` agricultural) |
| RES                                        | generators (`PV` → solar, `Wind` → wind, …; `subtype` keeps the SimBench type) |
| storage units `PV_Storage`, `PV_Storage_MV` (PV systems with home battery) | generators of type `solar`, subtype `PV_Storage`, `PV_Storage_MV` |
| other storage units (only in HV and EHV grids) | storage units                                            |
| profiles × `pLoad`, `pRES`, `pStor`, …     | active and reactive power time series                        |

**PV systems with home battery.** SimBench models them as storage units, but
their profile is the power at the grid connection of PV system *and* battery
together (SimBench documentation, section 4.1.5): it is never negative and
peaks at noon. As eDisGo storage units, the optimisation would dispatch them as
batteries and lose the PV energy. They are therefore generators in eDisGo; the
power flow is the same.

Conventional power plants, shunts, three-winding transformers and measurements
are not imported; MV grids and their LV grids do not contain them anyway.

## 5 What is in the grid?

### Topology

The MV grid is drawn dark, the LV grids light. SimBench places each LV grid at its
MV/LV substation. Open switch disconnectors, where the MV rings are split, are
marked red.

In [ ]:
def plot_grid(edisgo, ax, title=""):
    buses = edisgo.topology.buses_df
    lines = edisgo.topology.lines_df
    in_lv = buses.loc[lines.bus0, "v_nom"].values < 1
    for selection, color, width, label in [
        (in_lv, "#9ecae1", 0.4, "LV lines"),
        (~in_lv, "#08306b", 1.2, "MV lines"),
    ]:
        selected = lines[selection]
        segments = list(
            zip(
                buses.loc[selected.bus0, ["x", "y"]].values,
                buses.loc[selected.bus1, ["x", "y"]].values,
            )
        )
        ax.add_collection(
            LineCollection(segments, colors=color, linewidths=width, label=label)
        )
    station = edisgo.topology.mv_grid.station
    ax.scatter(station.x, station.y, s=120, marker="s", c="black", label="HV/MV station")
    switches = buses.loc[edisgo.topology.switches_df.bus_open]
    ax.scatter(switches.x, switches.y, s=25, c="red", label="open switch", zorder=3)
    ax.autoscale()
    ax.set_aspect(1 / 0.6)  # roughly equal distances at 53° latitude
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


fig, ax = plt.subplots(figsize=(8, 8))
plot_grid(edisgo, ax, "SimBench rural MV grid with all LV grids")
ax.legend(loc="lower right");

In [ ]:
topology = edisgo.topology
lv_buses = topology.buses_df.lv_grid_id.notna()
pd.Series(
    {
        "LV grids": len(topology._lv_grid_ids),
        "buses MV / LV": f"{(~lv_buses).sum()} / {lv_buses.sum()}",
        "lines MV / LV": f"{topology.mv_grid.lines_df.shape[0]} / "
        f"{topology.lines_df.shape[0] - topology.mv_grid.lines_df.shape[0]}",
        "MV line length in km": round(topology.mv_grid.lines_df.length.sum(), 1),
        "MV/LV transformers": len(topology.transformers_df),
        "switch disconnectors": len(topology.switches_df),
    },
    name="1-MVLV-rural-all-2-sw",
).to_frame()

### Loads and generators

Scenario 2 contains heat pumps and home charging points as own load types, so
eDisGo functions that look at them (e.g. `topology.charging_points_df`) work on
them directly.

In [ ]:
loads = topology.loads_df.groupby(["type", "sector"]).p_set.agg(
    number="size", nominal_power_MW="sum"
)
generators = topology.generators_df.groupby(["type", "subtype"]).p_nom.agg(
    number="size", nominal_power_MW="sum"
)
display(loads.round(2), generators.round(2))

The generator type `others` with subtype `lv_RES` and the conventional loads of
sector `other` are LV grids that SimBench represents as one generator or load at
an MV bus. In `1-MVLV-rural-all` only few of them are left, as almost all LV grids
are modelled in detail.

The heat pump profiles tell the heat source (`Air`, `Soil`) and the operating
mode, the charging point profiles the charging power in kW:

In [ ]:
simbench_loads = pd.read_csv(
    os.path.join(SIMBENCH_DIR, GRID.format(scenario=2), "Load.csv"), sep=";"
)
sector_coupling = simbench_loads[
    simbench_loads.profile.str.match("Air_|Soil_|HLS_")
]
sector_coupling.groupby("profile").pLoad.agg(number="size", mean_kW=lambda p: 1e3 * p.mean()).round(1)

### Time series

The time series are the SimBench profiles scaled with the nominal power of each
component. Below: the total power of each component type in the winter and the
summer week. Heat pumps and charging points add load peaks in winter mornings
and evenings, PV dominates in summer. PV systems with home battery feed in less
at noon and a little at night, as their battery shifts part of the PV energy.

In [ ]:
ts = edisgo.timeseries
loads_df = topology.loads_df
generators_df = topology.generators_df
pv_battery = generators_df.subtype.str.startswith("PV_Storage")


def total(df):
    return df.sum(axis=1)


by_type = pd.DataFrame(
    {
        "conventional loads": total(ts.loads_active_power[loads_df.index[loads_df.type == "conventional_load"]]),
        "heat pumps": total(ts.loads_active_power[loads_df.index[loads_df.type == "heat_pump"]]),
        "charging points": total(ts.loads_active_power[loads_df.index[loads_df.type == "charging_point"]]),
        "generators": -total(ts.generators_active_power[generators_df.index[~pv_battery]]),
        "PV with home battery": -total(ts.generators_active_power[generators_df.index[pv_battery]]),
    }
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, week, title in [(axes[0], winter, "winter week"), (axes[1], summer, "summer week")]:
    by_type.loc[week].plot(ax=ax, linewidth=1)
    total(by_type.loc[week]).plot(ax=ax, color="black", linewidth=1.5, label="residual load")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_title(title)
    ax.set_ylabel("MW (load positive, feed-in negative)")
axes[1].legend(loc="lower left", fontsize=8)
axes[0].get_legend().remove()
plt.tight_layout()

## 6 Power flow analysis

`analyze` runs a non-linear power flow and stores voltages, currents and power
flows in `edisgo.results`. The slack is the MV side of the HV/MV substation.

A power flow of this grid with about 5 500 buses takes roughly half a second per
time step. Grid issues occur in extreme situations, so instead of all 1 344 time
steps we analyse the critical ones: the 10 time steps with the highest residual
load (load minus generation; the *load case*) and the 10
with the lowest (the *feed-in case*). For a whole year,
`tools.temporal_complexity_reduction.get_most_critical_time_steps` selects
critical time steps in a more refined way.

In [ ]:
def critical_time_steps(edisgo, number=10):
    residual_load = edisgo.timeseries.residual_load
    return residual_load.nlargest(number).index.union(
        residual_load.nsmallest(number).index
    )


critical = critical_time_steps(edisgo)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
residual_load = edisgo.timeseries.residual_load
for ax, week, title in [(axes[0], winter, "winter week"), (axes[1], summer, "summer week")]:
    residual_load.loc[week].plot(ax=ax, color="black", linewidth=1)
    selected = residual_load.loc[critical.intersection(week)]
    ax.scatter(selected.index, selected, color="red", zorder=3, label="critical time steps")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_title(title)
    ax.set_ylabel("residual load in MW")
axes[0].legend()
plt.tight_layout()

In [ ]:
edisgo.analyze(timesteps=critical)

relative_load = check_tech_constraints.lines_relative_load(edisgo).max()
mv_lines = topology.mv_grid.lines_df.index
v = edisgo.results.v_res
lv_bus_names = topology.buses_df.index[lv_buses]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(
    [relative_load[mv_lines], relative_load.drop(mv_lines)],
    bins=30,
    label=["MV lines", "LV lines"],
    log=True,
)
axes[0].axvline(1, color="red", linewidth=0.8, label="allowed load")
axes[0].set_xlabel("maximum relative line load")
axes[0].set_ylabel("number of lines")
axes[0].legend()
axes[1].hist(
    [v[lv_bus_names].min(), v[lv_bus_names].max()],
    bins=30,
    label=["lowest voltage", "highest voltage"],
    log=True,
)
axes[1].set_xlabel("voltage at LV buses in p.u.")
axes[1].set_ylabel("number of buses")
axes[1].legend()
plt.tight_layout()

`check_tech_constraints` compares the results with the allowed line and
transformer loading and the allowed voltage band from the eDisGo config files and
lists every component that violates them. The most loaded lines:

In [ ]:
relative_load.sort_values(ascending=False).head(10).to_frame(
    "max. relative load"
).join(topology.lines_df[["type_info", "length", "kind"]]).round(2)

In [ ]:
def grid_issues(edisgo):
    relative_load = check_tech_constraints.lines_relative_load(edisgo).max()
    mv_lines = edisgo.topology.mv_grid.lines_df.index
    voltage = {
        level: len(check_tech_constraints.voltage_issues(edisgo, voltage_level=level))
        for level in ["mv", "mv_lv", "lv"]
    }
    return pd.Series(
        {
            "overloaded MV lines": (relative_load[mv_lines] > 1).sum(),
            "overloaded LV lines": (relative_load.drop(mv_lines) > 1).sum(),
            "overloaded MV/LV stations": len(
                check_tech_constraints.mv_lv_station_max_overload(edisgo)
            ),
            "MV buses with voltage issues": voltage["mv"],
            "MV/LV stations with voltage issues": voltage["mv_lv"],
            "LV buses with voltage issues": voltage["lv"],
            "highest relative line load": round(relative_load.max(), 2),
        }
    )


grid_issues(edisgo).to_frame("scenario 2 (2034)")

## 7 2016 versus 2034

The same grid in scenario 0 has no heat pumps, charging points or home batteries
and less PV. Comparing both shows what the energy transition in SimBench does to this
grid.

In [ ]:
edisgo_2016 = EDisGo()
edisgo_2016.import_simbench_grid(
    os.path.join(SIMBENCH_DIR, GRID.format(scenario=0)), timeindex=timeindex
)
critical_2016 = critical_time_steps(edisgo_2016)
edisgo_2016.analyze(timesteps=critical_2016)

pd.DataFrame(
    {
        "scenario 0 (2016)": grid_issues(edisgo_2016),
        "scenario 2 (2034)": grid_issues(edisgo),
    }
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, week, title in [(axes[0], winter, "winter week"), (axes[1], summer, "summer week")]:
    edisgo_2016.timeseries.residual_load.loc[week].plot(ax=ax, label="scenario 0 (2016)")
    edisgo.timeseries.residual_load.loc[week].plot(ax=ax, label="scenario 2 (2034)")
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.set_title(title)
    ax.set_ylabel("residual load in MW")
axes[0].legend()
plt.tight_layout()

## 8 Grid reinforcement

`reinforce` resolves all overloading and voltage issues with conventional grid
expansion (parallel or larger lines, additional transformers, splitting feeders)
and calculates the costs. We reinforce for the critical time steps of each
scenario.

In [ ]:
costs = {}
for name, obj, steps in [
    ("scenario 0 (2016)", edisgo_2016, critical_2016),
    ("scenario 2 (2034)", edisgo, critical),
]:
    obj.reinforce(timesteps_pfa=steps)
    costs[name] = obj.results.grid_expansion_costs.groupby("voltage_level").total_costs.sum()
costs = pd.DataFrame(costs).reindex(["mv", "mv/lv", "lv"]).fillna(0)
costs.loc["total"] = costs.sum()
costs.round(1).rename_axis("grid expansion costs in k€")

In [ ]:
changes = edisgo.results.equipment_changes
component = changes.index.map(
    lambda name: "line" if name in edisgo.topology.lines_df.index else "transformer"
)
changes.groupby([component, "change"]).size().rename("number").to_frame()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (name, obj) in zip(axes, [("scenario 0 (2016)", edisgo_2016), ("scenario 2 (2034)", edisgo)]):
    plot_grid(obj, ax, name)
    lines = obj.topology.lines_df
    reinforced = lines.loc[obj.results.grid_expansion_costs.index.intersection(lines.index)]
    buses = obj.topology.buses_df
    ax.add_collection(
        LineCollection(
            list(zip(buses.loc[reinforced.bus0, ["x", "y"]].values, buses.loc[reinforced.bus1, ["x", "y"]].values)),
            colors="orange",
            linewidths=2.5,
            label="reinforced lines",
        )
    )
axes[1].legend(loc="lower right")
plt.tight_layout()

## 9 What else can you do?

Once imported, a SimBench grid is an ordinary eDisGo object. Everything that
works on the topology and the time series works on it, for example:

| task | eDisGo |
|------|--------|
| power flow for single grids or voltage levels | `analyze(mode="mv")`, `analyze(mode="lv", lv_grid_id=1108)` |
| most critical time steps of a year | `tools.temporal_complexity_reduction.get_most_critical_time_steps` |
| grid reinforcement and costs | `reinforce()`, `results.grid_expansion_costs` |
| n-1 security of the MV grid | `reinforce(n_minus_one=True)` |
| reduce the grid size for optimisation | `spatial_complexity_reduction()` |
| add own components, e.g. more heat pumps or charging points | `add_component`, `integrate_component_based_on_geolocation` |
| own scenarios: scale time series | `timeseries.scale_timeseries` |
| optimal power flow with flexibilities | `pm_optimize` (needs Julia) |
| plots and maps | `plot_mv_line_loading`, `plot_plotly`, `histogram_voltage` |
| save and load results | `save`, `edisgo.edisgo.import_edisgo_from_files` |

### Limits of the SimBench data

* **Flexibility of heat pumps and charging points.** SimBench gives their
  electrical load profiles, not the heat demand, COP, thermal storage or
  charging events. eDisGo's heat pump operating strategies
  (`apply_heat_pump_operating_strategy`), charging strategies
  (`apply_charging_strategy`) and their flexibility in the optimisation need
  these data, so SimBench heat pumps and charging points are inflexible loads
  unless you add the data yourself.
* **Home batteries** are part of the feed-in profile of PV systems with home
  battery. To study battery operation, e.g. with eDisGo's optimisation, model
  the batteries as storage units yourself (SimBench gives their size in
  `Storage.csv`, column `eStore`) and give the PV systems a plain PV profile.
* **Only grids with an MV grid** can be imported (codes `MV` and `MVLV`). LV-only
  grids (`1-LV-…`) have no HV/MV transformer and eDisGo needs an MV grid.
* **Gas and heating grids** of SimBench Sektor are not covered; they had not been
  published when the import was written.
* SimBench's voltage limits per node and its study cases (`hL`, `lW`, …) are not
  used; eDisGo uses its own limits from the config files and the time series.